In [ ]:
import pandas as pd
import time

from sequences.get_sequences import get_sequences_by_possession, get_corners_sequences
from dataframe_modify.add_positions import add_positions
from dataframe_modify.add_context import add_context
from get_data.get_data import get_merged_events, get_sb360
from get_data.get_season_data import *
from corners.metric import *
from corners.get_dataframe import *

In [ ]:
params = {
    'pass_to_zone_I': -0.8,
    'pass_to_zone_II': 0.1,
    'pass_to_zone_III': 0.2,
    'pass_incomplete': -0.5,
    'pass_shot_assist': 0.4,
    'pass_goal_assist': 0.4,
    'pass_cross': 0.2,
    'pass_by_opponent': -0.5,
    'shot_goal': 2,
    'shot_saved': 1,
    'shot_blocked': 0.5,
    'shot_off_target': 0.5,
    'shot_post': 1,
    'shot_saved_to_post': 1,
    'shot_saved_off_target': 0.5,
    'clearance_by_opponent': -0.3,
    'ball_recovery_by_opponent': -0.3,
    'ball_recovery_by_own_team': 0.5,
    'foul_committed_by_own_team': -0.5,
    'foul_committed_by_own_team_with_card': -1,
    'foul_committed_by_opponent': 0.5,
    'foul_committed_by_opponent_with_penalty': 1,
}

In [ ]:
# match = 3836371 #Lech Poznan - Lechia Gdansk
# match = 3836312 # Lech Poznań - Korona Kielce
match = 3788776
def_areas_x = 12
def_areas_y = 8

events_types_to_analize = ['passes', 'carrys', 'ball_receipts', 'ball_recoverys', 'duels', 'dribbles', 'blocks', 'clearances', 
'dispossesseds', 'errors', 'foul_committeds', 'half_ends', 'interceptions', 'miscontrols', 'shots', 'goal_keepers']

events = get_merged_events(events_types_to_analize, match)

start_time = time.time()
frames = get_sb360(match)
events = add_context(events, frames)
events = add_positions(events)
sequences = get_sequences_by_possession(events)
corners = get_corners_sequences(sequences)

In [ ]:
corners =  [evaluate_corner(corner, params) for corner in corners]

In [ ]:
def df_print_corner(df):
    corner_df = df[['period', 'timestamp', 'type', 'team', 'player', 'event', 'corner_score']]
    return corner_df

In [ ]:
df_print_corner(corners[-2])#.to_csv('corner_2.csv', index=False)

In [ ]:
corners[-1].columns

In [ ]:
def get_corner_obv(corner):
    obv = 0
    # sum obv_total_net for all events in corner
    for i in range(len(corner)):
        if corner.iloc[i]['obv_for_net']:
            obv += float(corner.iloc[i]['obv_for_net'])
    return obv

In [ ]:
get_corner_obv(corners[-1])

In [ ]:
corners[-1].to_csv('corner_1.csv', index=False)

In [ ]:
def get_corner_xg(corner):
    xg = 0
    for i in range(len(corner)):
        if corner.iloc[i]['type'] == 'Shot':
            if corner.iloc[i]['event'].get('statsbomb_xg') != None:
                print(corner.iloc[i]['event'].get('statsbomb_xg'))
                xg += float(corner.iloc[i]['event'].get('statsbomb_xg'))
    return xg

In [ ]:
get_corner_xg(corners[-1])

In [ ]:
print(corners[-1]['event'].iloc[6].get('statsbomb_xg'))

In [ ]:
import xgboost as xgb


In [ ]:
corners[-1].columns

In [ ]:
xgb_df = pd.DataFrame()
xgb_df.append([get_df_row(corner) for corner in corners])

In [ ]:
xgb_df